# Hedging at Scale Workflow

This example shows how to build and size a set of hedges for an existing portfolio with the Axioma API,
and how to do it at scale: the hedging instruments are created on the fly from templates, with maturities
that match the risk factors they are meant to hedge.

The workflow is:

1. **Decompose the risk** of the unhedged portfolio into risk type, factor type and risk factor.
2. **Create the hedging instruments** from the risk factors found and load them into a hedging portfolio:
   * constant maturity bonds for the interest rate (`GVT`) factors,
   * zero coupon inflation swaps for the inflation (`BEI`) factors,
   * FX forwards for the currency factors.
3. **Calibrate the OTC hedges** (break-even swap rate, fair forward rate) so that they have a zero PV at inception.
4. **Calculate the hedge quantities** with the `AxR-Hedging Quantities` statistic and a risk factor reduction rule.
5. **Update the hedging portfolio** with the calculated quantities.

Three portfolios are used:

| Portfolio | Role |
| --- | --- |
| `<portfolio_name>` | The unhedged portfolio. It must already exist and have a default currency. |
| `<portfolio_name>_HEDGING` | The hedging universe. Created if needed; its positions on the position date are flushed at the start of each run. |
| `<portfolio_name>_HEDGED` | The hedged portfolio. Created if needed; it holds the unhedged and hedging portfolios as positions so the combined risk can be analysed. |

The same workflow is available as a script: [hedging_at_scale_workflow_example.py](./hedging_at_scale_workflow_example.py).

## Imports

### Logging

The standard logging library is used to log from within the package.
In order to see the logging statements you must add at least one handler.
In this case we will just log to the stdout. The level will be info.

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO, 
    format=r'%(asctime)s %(levelname)-8s %(message)s', 
    datefmt='%Y-%m-%d %H:%M:%S'
)

### Other common imports

In [2]:
import time
import pandas as pd
from io import StringIO
import json
import re
import datetime

### Importing from axiomapy

`AxiomaSession` manages the connection to the API and is used for every request.
The wrappers used in this example are `PortfoliosAPI` (portfolios and positions), `TemplatesAPI` (creating instruments from templates)
and `AnalysesAPI`/`AnalysesRiskAPI` (submitting analyses and reading their status, logs and results).
The `utils` module has helpers that submit an analysis, poll until it finishes and return the results as a pandas DataFrame.

In [3]:
from axiomapy import AxiomaSession
from axiomapy.axiomaapi import AnalysesAPI, AnalysesRiskAPI, PortfoliosAPI, TemplatesAPI, utils, enums
from axiomapy.odatahelpers import oDataFilterHelper as od
from axiomapy import axiomaexceptions

## Helper functions

The following functions wrap the individual steps of the workflow. They are defined first and used in the workflow section further down.

Note: where the script version calls `exit()` after a failed request, the notebook raises an exception instead.
This stops the cell with a traceback but keeps the kernel (and the session) alive.

### Risk decomposition report

`run_risk_decomp_report` builds an ad hoc analysis definition with three aggregation levels (Risk Type, Factor Type, Risk Factor)
and the parametric `AxR-Risk Exposure (P-Shell)` statistic, submits it as a portfolio analysis on the unhedged portfolio and polls until it completes (up to one hour).
The results are returned as a DataFrame with one row per risk factor, for example `CH.CHF.GVT.ZC : 6M` or `GB.GBP.BEI : 10Y`.

Custom risk settings and a custom risk statistic can be passed; by default the API defaults are used.

In [4]:
def run_risk_decomp_report(unhedged_portfolio_id, position_date, data_partition, pricing_source, risk_settings="Default", risk_statistic="Default"):
    # Break down the portfolio to risk factors
    risk_decomp_json = {
        "name": "Risk Decomposition",
        "aggregationLevelDefinitions": [
            {
                "name": "Risk Type",
                "item": {
                    "templateName": "Risk Type",
                    "content": {}
                }
            },
            {
                "name": "Factor Type",
                "item": {
                    "templateName": "Factor Type",
                    "content": {}
                }
            },
            {
                "name": "Risk Factor",
                "item": {
                    "templateName": "Risk Factor",
                    "content": {}
                }
            }
        ],
        "statisticDefinitions": [
            {
                "name": "Risk Exposure (Portfolio)",
                "item": {
                    "templateName": "AxR-Risk Exposure (P-Shell)",
                    "content": {
                        "Description": "Parametric portfolio risk exposure using MAC Global resolution and expressed in monetary terms"
                    }
                }
            }
        ]
    }

    if risk_settings != "Default":
        risk_decomp_json["statisticDefinitions"][0]["item"]["content"]["RiskSettings"] = risk_settings
    if risk_settings != "Default":
        risk_decomp_json["statisticDefinitions"][0]["item"]["content"]["RiskStatistic"] = risk_statistic

    analysis_payload = {
        "analysisDate": position_date,
        "positionDate": position_date,
        "analysisDefinition": risk_decomp_json,
        "dataPartition": data_partition,
        "riskDataSource": pricing_source,
        "aggregationOptions": {
            "aggregate": "OnCompletion",
            "compute": "MissingOnly"
        },
        "showFactorModelNameAsRiskFactorPrefix": True
    }


    status, headers = utils.request_aggregation(analysis_payload, unhedged_portfolio_id, timelimit=3600)
    risk_decomp_df = utils.get_results_and_logs(request_id=int(headers.headers["location"].split("/")[-1]),
                                          stat=status.json()["status"])
    return risk_decomp_df

### Break-even report

`run_breakeven_report` prices a set of positions passed inline (no saved portfolio is needed) with the `AxR-Coverage` and `AxR-Breakeven` statistics, aggregated by `ClientId`.
It is used to find the swap rate of the inflation swaps and the forward rate of the FX forwards that give a zero PV at inception.

In [5]:
def run_breakeven_report(swaps_positions, position_date, data_partition, pricing_source):
    # Calculate the breakeven (swap) rate of the OTC swaps
    break_even_json = {
        "name": "Break-even rate",
        "aggregationLevelDefinitions": [
            {
                "name": "ClientId",
                "item": {
                    "templateName": "{DefaultTemplateFor_ViewReportingLevelOnPosition}",
                    "content": {
                        "AttributeName": "ClientId"
                    }
                }
            }
        ],
        "statisticDefinitions": [
            {
                "name": "Coverage",
                "item": {
                    "templateName": "AxR-Coverage"
                }
            },
            {
                "name": "breakeven",
                "item": {
                    "templateName": "AxR-Breakeven"
                }
            }
        ]
    }

    analysis_payload = {
        "portfolioName": "Swaps Portfolio",
        "analysisDate": position_date,
        "positionDate": position_date,
        "analysisDefinition": break_even_json,
        "dataPartition": data_partition,
        "riskDataSource": pricing_source,
        "aggregationOptions": {
            "aggregate": "OnCompletion",
            "compute": "MissingOnly"
        },
        "positions": swaps_positions
    }


    status, headers = utils.request_position_analytics(analysis_payload, timelimit=3600)
    break_even_df = utils.get_results_and_logs(request_id=int(headers.headers["location"].split("/")[-1]),
                                          stat=status.json()["status"])
    return break_even_df

### Interest rate hedges

`create_ir_hedges` selects the interest rate risk factors whose name contains `GVT` from the risk decomposition.
For each one it derives the currency and the tenor from the factor name and upserts a constant maturity bond (`Modeling_Of_Bond` template)
whose maturity is a fixed tenor from the analysis date, so the bond always matches the risk factor it hedges.
The bonds are then loaded into the hedging portfolio with a placeholder quantity of 1; the quantities are calculated later in the workflow.

In [6]:
def create_ir_hedges(risk_decomp_df, hedging_portfolio_name, hedging_portfolio_id, position_date):
    # Create constant maturity bonds for duration (GVT interest rate risk factors) hedging
    mask = (risk_decomp_df['RiskType'] == 'Risk Type : Interest Rate') & (
        risk_decomp_df['RiskFactor'].str.contains('GVT', na=False))
    filtered_df = risk_decomp_df.loc[mask]
    unique_ir_risk_factors = filtered_df['RiskFactor'].unique().tolist()
    print("Interest Rate Risk Factors to hedge:")
    print(unique_ir_risk_factors)

    if unique_ir_risk_factors:
        hedging_positions_list = []
        for ir_risk_factor in unique_ir_risk_factors:
            left, right = ir_risk_factor.split(':')
            currency = left.strip().split('.')[1]
            match = re.match(r'(\d+)([A-Za-z]+)', right.strip())
            if not match:
                raise ValueError(f"Invalid tenor format: {right.strip()}")
            tenor = match.group(1)
            tenor_unit = match.group(2)

            tenor_unit_map = {"D": "Day", "W": "Week", "M": "Month", "Y": "Year"}

            zc_constant_maturity_bond = {
                "$type": "TemplatedRawDataInputs",
                "templateName": "{DefaultTemplateFor_Modeling_Of_Bond}",
                "content": {
                    "$type": "Modeling_Of_Bond",
                    "Name": f"Constant_Maturity_{currency}_{tenor}{tenor_unit}",
                    "termsAndConditions": {
                        "$type": "TemplatedRawDataInputs",
                        "templateName": "{DefaultTemplateFor_Bond}",
                        "content": {
                            "$type": "Bond",
                            "Name": f"Constant_Maturity_{currency}_{tenor}{tenor_unit}",
                            "reportingAttributes": {
                                "reportingAttributes": []
                            },
                            "accruingPeriodConvention": None,
                            "businessConventions": {
                                "businessDayConvention": "NoAdjustment",
                                "holidays": {
                                    "$type": "HolidaysDefinitions.NoHolidays"
                                }
                            },
                            "couponSchedule": {
                                "$type": "CouponSchedule.FixedTenor",
                                "fixedTenorFromAnalysisDate": {
                                    "numberOfUnits": tenor,
                                    "unit": tenor_unit_map[tenor_unit]
                                }
                            },
                            "currency": currency,
                            "dayCountConvention": "D30Y360US",
                            "exDividend": None,
                            "expectedMaturityDate": None,
                            "firstCouponStartAccruingDate": None,
                            "issueDate": None,
                            "issuerId": None,
                            "notional": 100,
                            "recoveryRate": {
                                "inScale": "Absolute",
                                "number": 0.4
                            },
                            "redemptionValue": 100,
                            "amountOutstandingHistory": None,
                            "couponSettlementConventions": {
                                "businessDayConvention": "NoAdjustment",
                                "holidays": {
                                    "$type": "HolidaysDefinitions.NoHolidays"
                                }
                            },
                            "extendedMaturityDate": None,
                            "isAmortizing": False,
                            "isFlatTrading": False,
                            "pIKTerms": None,
                            "priceTimeSeriesId": None,
                            "settlementPeriod": {
                                "numberOfUnits": 0,
                                "unit": "BusinessDay"
                            },
                            "sinkingSchedule": {
                                "rates": None,
                                "isFundingProvision": False,
                                "sinkingRatesScale": "Absolute"
                            },
                            "status": "None"
                        }
                    },
                    "modelingAssumptions": {
                        "$type": "TemplatedRawDataInputs",
                        "templateName": "{DefaultTemplateFor_Bond_ValuationThrough_SpotMarketExpectationsModel}",
                        "content": {
                            "$type": "Bond_ValuationThrough_SpotMarketExpectationsModel",
                            "Name": f"Constant_Maturity_{currency}_{tenor}{tenor_unit}",
                            "dealContext": {
                                "creditModels": None,
                                "isFXHedged": False,
                                "riskFreeCurves": [],
                                "discountCurves": [],
                                "cTDDiscountCurves": None,
                                "riskFreeReferenceDiscountCurves": None
                            },
                            "valuationMethod": None,
                            "calibrationMethods": {
                                "base": {
                                    "$type": "SpotMarketExpectationsModel.Calibrator",
                                    "interestRateCurveInterpolationMethod": "PieceWiseLinear",
                                    "ultimateForwardRate": None,
                                    "lastTenorOverride": None
                                },
                                "local": None,
                                "priceReconciliation": None
                            }
                        }
                    }
                }
            }

            r = TemplatesAPI.patch_entities(template_name="{DefaultTemplateFor_Modeling_Of_Bond}",
                                            entities_upsert=[zc_constant_maturity_bond])

            if r.status_code == 204:
                print(f"Constant_Maturity_{currency}_{tenor}{tenor_unit} Created")
                hedging_positions_list.append(f'Constant_Maturity_{currency}_{tenor}{tenor_unit}')
            else:
                print(
                    f"Error: Received status code {r.status_code} when creating Constant_Maturity_{currency}_{tenor}{tenor_unit}")
                raise RuntimeError(f"Received status code {r.status_code} - stopping the workflow")
    my_ir_hedges = []
    for cmb in hedging_positions_list:
        my_ir_hedges.append(
            {
                "clientId": cmb,
                "description": cmb,
                "identifiers": [
                    {
                        "type": "ClientGiven",
                        "value": cmb
                    }
                ],
                "quantity": {
                    "value": 1,
                    "scale": "NumberOfInstruments"
                }
            }
        )

    r = PortfoliosAPI.patch_positions(portfolio_id=hedging_portfolio_id,
                                      as_of_date=position_date,
                                      positions_upsert=my_ir_hedges,
                                      positions_remove=[])
    print(f'{hedging_portfolio_name} positions imported')

    return my_ir_hedges

### Inflation hedges

`create_ie_hedges` selects the inflation risk factors of the given currency (for example `GB.GBP.BEI : 10Y` for `GBP`).
For each one it upserts a zero coupon inflation swap (`AxR-OTC-ZC Inflation Swap` template) on the UK RPI index,
starting on the position date and maturing after the tenor of the factor, first with a placeholder swap rate of 1%.
The swaps are loaded into the hedging portfolio, the break-even report returns their fair swap rates,
and the swaps are patched again with those rates so they have a zero PV at inception.

In [7]:
def create_ie_hedges(risk_decomp_df, currency, hedging_portfolio_name, hedging_portfolio_id, position_date, data_partition="AxiomaUS", pricing_source="Default"):
    def patch_zcils(ie_risk_factor, swap_rate_map):
        left, right = ie_risk_factor.split(':')
        currency = left.strip().split('.')[1]
        match = re.match(r'(\d+)([A-Za-z]+)', right.strip())
        if not match:
            raise ValueError(f"Invalid tenor format: {right.strip()}")
        tenor = match.group(1)
        tenor_unit = match.group(2)

        year_to_add = 0
        month_to_add = 0
        if tenor_unit == 'Y':
            year_to_add = int(tenor)
        elif tenor_unit == 'M':
            month_to_add = int(tenor)
        start = datetime.datetime.strptime(position_date, '%Y-%m-%d')
        end = start + pd.DateOffset(years=year_to_add, months=month_to_add)
        swap_name = f"ZC_Inflation_Swap_{currency}_{tenor}{tenor_unit}"

        zc_inflation_swap = {
        "templateName": "AxR-OTC-ZC Inflation Swap",
        "content": {
            "Name": swap_name,
            "currency": f"Currency={currency}",
            "inflation_lag": 2,
            "inflation_lag_unit": "Month",
            "inflation_market_convention": "UK",
            "inflation_index": "Ticker=UKRPI Index",
            "start_date": start.strftime('%Y-%m-%dT%H:%M:%S'),
            "maturity_date": end.strftime('%Y-%m-%dT%H:%M:%S'),
            "swap_rate": swap_rate_map.get(swap_name, 0.01)
        }
    }

        r = TemplatesAPI.patch_entities(template_name="{DefaultTemplateFor_Modeling_Of_ZeroCouponInflationSwap}",
                                        entities_upsert=[zc_inflation_swap])

        if r.status_code == 204:
            swap_name = f'ZC_Inflation_Swap_{currency}_{tenor}{tenor_unit}'
            print(f"{swap_name} Updated")
        else:
            print(
                f"Error: Received status code {r.status_code} when creating ZC_Inflation_Swap_{currency}_{tenor}{tenor_unit}")
            raise RuntimeError(f"Received status code {r.status_code} - stopping the workflow")
        return swap_name

    # Create zero coupon inflation swap for inflation (GBP.BEI risk factors) hedging
    mask = (risk_decomp_df['RiskType'] == 'Risk Type : Inflation') & (
        risk_decomp_df['RiskFactor'].str.contains(currency, na=False))
    filtered_df = risk_decomp_df.loc[mask]
    unique_ie_risk_factors = filtered_df['RiskFactor'].unique().tolist()
    print("Inflation Risk Factors to hedge:")
    print(unique_ie_risk_factors)

    if unique_ie_risk_factors:
        hedging_positions_list = []
        for ie_risk_factor in unique_ie_risk_factors:
            swap_name = patch_zcils(ie_risk_factor, {})
            hedging_positions_list.append(swap_name)

        my_ie_hedges = []
        for zcils in hedging_positions_list:
            my_ie_hedges.append(
                {
                    "clientId": zcils,
                    "description": zcils,
                    "identifiers": [
                        {
                            "type": "ClientGiven",
                            "value": zcils
                        }
                    ],
                    "quantity": {
                        "value": 1,
                        "scale": "NumberOfInstruments"
                    }
                }
            )

        r = PortfoliosAPI.patch_positions(portfolio_id=hedging_portfolio_id,
                                          as_of_date=position_date,
                                          positions_upsert=my_ie_hedges,
                                          positions_remove=[])
        print(f'{hedging_portfolio_name} positions imported')

        # Calculate the break-even (swap) rate so the PV of these swaps are 0 at initiation
        break_even_df = run_breakeven_report(my_ie_hedges, position_date, data_partition, pricing_source)
        swap_rate_map = break_even_df[break_even_df['ClientId'] != 'Total'].set_index('ClientId')['breakeven'].to_dict()

        for ie_risk_factor in unique_ie_risk_factors:
            swap_name = patch_zcils(ie_risk_factor, swap_rate_map)

        return my_ie_hedges

### FX hedges

`create_fx_hedges` creates, for each currency to hedge, a one month FX forward (`AxR-OTC-FX Forward` template)
selling the currency to hedge and buying the portfolio currency, first with a placeholder forward rate of 1.
The forwards are loaded into the hedging portfolio, the break-even report returns their fair forward rates,
and the forwards are patched again with those rates so they have a zero PV at inception.

In [8]:
def create_fx_hedges(currencies_list_to_hedge, portfolio_currency, hedging_portfolio_name, hedging_portfolio_id, position_date, data_partition="AxiomaUS", pricing_source="Default"):
    def patch_fxforward(currency_to_hedge, portfolio_currency, fx_rate_map):
        start = datetime.datetime.strptime(position_date, '%Y-%m-%d')
        end = start + pd.DateOffset(months=1)
        fx_forward_name = f"FX_Forward_{currency_to_hedge}{portfolio_currency}_{end.strftime('%Y-%m-%d')}"

        fx_forward = {
        "templateName": "AxR-OTC-FX Forward",
        "content": {
            "Name": fx_forward_name,
            "buy_currency": f"Currency={portfolio_currency}",
            "sell_currency": f"Currency={currency_to_hedge}",
            "maturity_date": end.strftime('%Y-%m-%dT%H:%M:%S'),
            "forward_rate": fx_rate_map.get(fx_forward_name, 1)
        }
    }

        r = TemplatesAPI.patch_entities(template_name="{DefaultTemplateFor_Modeling_Of_FXForward}",
                                        entities_upsert=[fx_forward])

        if r.status_code == 204:
            fx_forward_name = f"FX_Forward_{currency_to_hedge}{portfolio_currency}_{end.strftime('%Y-%m-%d')}"
            print(f"{fx_forward_name} Updated")
        else:
            print(
                f"Error: Received status code {r.status_code} when creating FX_Forward_{currency_to_hedge}{portfolio_currency}_{end.strftime('%Y-%m-%d')}")
            raise RuntimeError(f"Received status code {r.status_code} - stopping the workflow")
        return fx_forward_name

    # Create zero coupon inflation swap for inflation (GBP.BEI risk factors) hedging

    hedging_positions_list = []
    for currency_to_hedge in currencies_list_to_hedge:
        fx_forward_name = patch_fxforward(currency_to_hedge, portfolio_currency,{})
        hedging_positions_list.append(fx_forward_name)

    my_fx_hedges = []
    for fx_forward in hedging_positions_list:
        my_fx_hedges.append(
            {
                "clientId": fx_forward,
                "description": fx_forward,
                "identifiers": [
                    {
                        "type": "ClientGiven",
                        "value": fx_forward
                    }
                ],
                "quantity": {
                    "value": 1,
                    "scale": "NumberOfInstruments"
                }
            }
        )

    r = PortfoliosAPI.patch_positions(portfolio_id=hedging_portfolio_id,
                                      as_of_date=position_date,
                                      positions_upsert=my_fx_hedges,
                                      positions_remove=[])
    print(f'{hedging_portfolio_name} positions imported')

    # Calculate the fx forward rate so the PV of these forwards are 0 at initiation
    break_even_df = run_breakeven_report(my_fx_hedges, position_date, data_partition, pricing_source)
    fx_forward_rate_map = break_even_df[break_even_df['ClientId'] != 'Total'].set_index('ClientId')['breakeven'].to_dict()

    for currency_to_hedge in currencies_list_to_hedge:
        fx_forward_name = patch_fxforward(currency_to_hedge, portfolio_currency, fx_forward_rate_map)


    return my_fx_hedges

## Logging in

Load the credentials from `credentials/credentials.json` (copy `sample.credentials.json` and fill in the details, see the [README](./README.md))
and create the session that is used for all the requests below.

In [9]:
# For testing load up some credentials
from load_credentials import get_user
user1 = get_user('user1')

Current working directory: C:\Users\RICH\PythonProject\axioma-py\axiomapy\examples
Looking for credentials.json at: C:\Users\RICH\PythonProject\axioma-py\axiomapy\examples\credentials\credentials.json
Loaded the credentials from the json..
user1: richan@qa-repro


In [ ]:
AxiomaSession.use_session(
    username=user1['username'],
    password=user1['password'],
    domain=user1['domain'],
    client_id=user1['client_id']
)

## Workflow parameters

Set the name of the unhedged portfolio, the position date, the data partition and the pricing source.
The names of the hedging and hedged portfolios are derived from the portfolio name.

`hedging_rule` is the risk factor reduction rule used by the `AxR-Hedging Quantities` statistic: it selects, by name, the risk factors the hedges must neutralise.
Several conditions can be combined with `|`. The rule must be consistent with the hedging instruments created further down:
`GVT` for the constant maturity bonds, `GB.GBP.BEI` for the inflation swaps and `Currency` for the FX forwards.

In [11]:
portfolio_name = "Your Portfolio"
hedging_portfolio_name = f"{portfolio_name}_HEDGING"
hedged_portfolio_name = f"{portfolio_name}_HEDGED"
position_date = '2026-05-01'
data_partition = 'AxiomaUS'
pricing_source = 'Default'

# Example hedging cases to hedge GVT factors for all currencies, GBP inflation factor or FX risk factors
# hedging_rule = "CONTAINS(#RiskFactorName,\"GVT\") | CONTAINS(#RiskFactorName,\"GB.GBP.BEI\") |
# hedging_rule = "CONTAINS(#RiskFactorName,\"GB.GBP.BEI\")"
hedging_rule = "CONTAINS(#RiskFactorName,\"Currency\")"

## Set up the portfolios

### Find the unhedged portfolio

The unhedged portfolio must already exist. Its id and default currency are read from the API.
The default currency is reused for the hedging and hedged portfolios and is the buy currency of the FX forwards.

In [ ]:
# Get the PortfolioId of the unhedged portfolio and make sure it has a default portfolio currency
try:
    response = PortfoliosAPI.get_portfolios(filter_results="contains(name, '" + portfolio_name + "')")
    portfolios_json = response.json()
    unhedged_portfolio_id = portfolios_json['items'][0]['id']
    default_currency = portfolios_json['items'][0]['defaultCurrency']
    print(f"Unhedged PortfolioId: {unhedged_portfolio_id}, Portfolio Currency: {default_currency}")

except (KeyError, IndexError, TypeError):
    raise ValueError(
        f"Portfolio with name '{portfolio_name}' not found or the portfolio does not have a default currency.")

### Add some bonds to the unhedged portfolio

For the purpose of the example a GBP inflation linked gilt, a USD corporate bond and a CHF bond are upserted into the unhedged portfolio on the position date,
so that there are interest rate, inflation and FX exposures to hedge. Skip this cell to work with the existing positions of the portfolio.

In [ ]:
# Add some bonds to the unhedged portfolio for the example
patch_position = [{"clientId": "UKTI 0.625 22-Mar-2040",
                   "identifiers": [{"type": "ISIN", "value": "GB00B3LZBF68"}],
                   "quantity": {"value": 10000,
                                "scale": "NotionalValue"}},
                  {"clientId": "DOW 5.25 15-Nov-2041",
                   "identifiers": [{"type": "ISIN", "value": "US260543CE14"}],
                   "quantity": {"value": 10000,
                                "scale": "NotionalValue"}},
                   {"clientId": "SWISS 4 06-Jan-2049",
                    "identifiers": [{"type": "ISIN", "value": "CH0009755197"}],
                    "quantity": {"value": 10000,
                                 "scale": "NotionalValue"}}]
patch = PortfoliosAPI.patch_positions(
    as_of_date=position_date,
    portfolio_id=unhedged_portfolio_id,
    positions_upsert=patch_position,
    positions_remove=[]
)

### Create the hedging portfolio

The hedging portfolio holds the hedging universe. It is created with the same default currency as the unhedged portfolio;
if it already exists its id is looked up instead. Any positions on the position date are deleted so that each run starts from an empty hedging universe.

In [ ]:
# Create a portfolio which will contain the hedging universe
hedging_portfolio = {
    "name": hedging_portfolio_name,
    "longName": hedging_portfolio_name,
    "defaultCurrency": default_currency,
}

try:
    r = PortfoliosAPI.post_portfolio(portfolio=hedging_portfolio)
    hedging_portfolio_id = int(r.headers["location"].split("/")[-1])
except (axiomaexceptions.AxiomaRequestError, axiomaexceptions.AxiomaRequestValidationError,
        axiomaexceptions.AxiomaRequestStatusError):
    filters = [
        od.equals("name", f"{portfolio_name}_HEDGING")
    ]
    filter_ = " ".join(filters)
    ptfs = PortfoliosAPI.get_portfolios(filter_results=filter_)
    hedging_portfolio_id = ptfs.json()['items'][0]['id']
print(f"Hedging PortfolioId: {hedging_portfolio_id}")

r = PortfoliosAPI.delete_positions(portfolio_id=hedging_portfolio_id, as_of_date=position_date)
print(f'{hedging_portfolio_name} positions flushed')

### Create the hedged portfolio

The hedged portfolio combines the unhedged and hedging portfolios. It is created (or looked up) in the same way and its positions on the position date are flushed.

In [ ]:
# Create a portfolio which will combine the unhedged and hedging portfolio
hedged_portfolio = {
    "name": hedged_portfolio_name,
    "longName": hedged_portfolio_name,
    "defaultCurrency": default_currency,
}

try:
    r = PortfoliosAPI.post_portfolio(portfolio=hedged_portfolio)
    hedged_portfolio_id = int(r.headers["location"].split("/")[-1])
except (axiomaexceptions.AxiomaRequestError, axiomaexceptions.AxiomaRequestValidationError,
        axiomaexceptions.AxiomaRequestStatusError):
    filters = [
        od.equals("name", f"{portfolio_name}_HEDGED")
    ]
    filter_ = " ".join(filters)
    ptfs = PortfoliosAPI.get_portfolios(filter_results=filter_)
    hedged_portfolio_id = ptfs.json()['items'][0]['id']
print(f"Hedged PortfolioId: {hedged_portfolio_id}")

r = PortfoliosAPI.delete_positions(portfolio_id=hedged_portfolio_id, as_of_date=position_date)
print(f'{portfolio_name}_HEDGED positions flushed')

The unhedged and hedging portfolios are then loaded as positions of the hedged portfolio, one unit each, using the `portfolio` identifier type.
The hedged portfolio can be analysed like any other portfolio to check the risk after hedging.

In [ ]:
my_positions = [
    {
        "clientId": "Unhedged Portfolio",
        "description": portfolio_name,
        "identifiers": [
            {
                "type": "portfolio",
                "value": portfolio_name
            }
        ],
        "quantity": {
            "value": 1,
            "scale": "NumberOfInstruments"
        }
    },
    {
        "clientId": "Hedging Portfolio",
        "description": hedging_portfolio_name,
        "identifiers": [
            {
                "type": "portfolio",
                "value": hedging_portfolio_name
            }
        ],
        "quantity": {
            "value": 1,
            "scale": "NumberOfInstruments"
        }
    }
]

r = PortfoliosAPI.patch_positions(portfolio_id=hedged_portfolio_id,
                                  as_of_date=position_date,
                                  positions_upsert=my_positions,
                                  positions_remove=[])

## Run the risk decomposition

The risk of the unhedged portfolio is decomposed into risk factors. This is the list of factors the hedges are built from.

In [ ]:
risk_decomp_df = run_risk_decomp_report(unhedged_portfolio_id, position_date, data_partition, pricing_source, risk_settings="Default", risk_statistic="Default")
risk_decomp_df

## Create the hedging instruments

Three hedging examples are available. Uncomment the ones to use and make sure the `hedging_rule` set in the parameters matches:

* **Duration hedging**: constant maturity bonds matching the `GVT` interest rate factors of all currencies.
* **Inflation hedging**: zero coupon inflation swaps matching the GBP inflation (`GB.GBP.BEI`) factors.
* **FX hedging**: one month FX forwards for the GBP and CHF exposures against the portfolio currency.

Each helper upserts the instruments, loads them into the hedging portfolio with a placeholder quantity and returns the positions, which are collected in `my_hedges`.

In [ ]:
my_hedges = []
if not risk_decomp_df.empty:
    # Duration hedging example using constant maturity bonds with matching maturities to hedge GVT interest rate risk factors
    # my_ir_hedges = create_ir_hedges(risk_decomp_df, portfolio_name, hedging_portfolio_id, position_date)
    # my_hedges.extend(my_ir_hedges)

    # # Inflation hedging (GBP) example using zero coupon inflation swap with matching maturities to hedge GBP inflation risk factors
    # my_ie_hedges = create_ie_hedges(risk_decomp_df, 'GBP', portfolio_name, hedging_portfolio_id, position_date, data_partition, pricing_source)
    # my_hedges.extend(my_ie_hedges)
    # FX hedging (GBP, CHF) example using FX forward with 1 month maturity to hedge FX risk factors
    my_fx_hedges = create_fx_hedges(['GBP', 'CHF'], default_currency, portfolio_name, hedging_portfolio_id, position_date,  data_partition, pricing_source)
    my_hedges.extend(my_fx_hedges)

## Calculate the hedge quantities

`utils.get_hedges_quantity` runs the `AxR-Hedging Quantities` statistic on the unhedged portfolio, with the hedging portfolio as the hedging universe
and `hedging_rule` as the risk factor reduction rule. The result is a single string of `instrument:quantity` pairs separated by `;`.

In [ ]:
# Get the hedging instruments' quantity needed to hedge the risk factors
hedges_quantity_df = utils.get_hedges_quantity(hedging_rule, unhedged_portfolio_id, hedging_portfolio_name, position_date, data_partition, pricing_source, "MissingOnly")
hedges_quantity_df

## Update the hedging portfolio

The quantities are parsed from the report and their sign is flipped so that the hedges offset the selected exposures instead of replicating them.
The quantity of each hedge position is updated and the hedging portfolio is patched.
The hedged portfolio now reflects the unhedged portfolio plus its hedges and can be analysed to check the remaining risk.

In [ ]:
# Update the quantity of the hedging instruments
hedging_dict = {}

hedging_results = hedges_quantity_df[f'HEDGING ({hedging_rule})'].iloc[0]
for pair in hedging_results.split(';'):
    key, value = pair.split(':')
    hedging_dict[key.strip()] = float(value) * -1

for hedge in my_hedges:
    client_id = hedge['clientId']
    if client_id in hedging_dict:
        hedge['quantity']['value'] = hedging_dict[client_id]

r = PortfoliosAPI.patch_positions(portfolio_id=hedging_portfolio_id,
                                  as_of_date=position_date,
                                  positions_upsert=my_hedges,
                                  positions_remove=[])
print(f'{hedging_portfolio_name} positions quantity updated')

The final hedge quantities:

In [ ]:
pd.DataFrame(
    [(hedge['clientId'], hedge['quantity']['value']) for hedge in my_hedges],
    columns=['clientId', 'quantity']
)

## Clean up

Close the session.

In [ ]:
AxiomaSession.current.close()